# Chat History 입력 및 요약

## 실습 목표
---
사람처럼 대화를 주고 받을 수 있도록 대화 기록 (Chat History) 을 저장하고 입력하는 기능을 구현하는 방법을 학습합니다.

## 실습 목차
---

1. **Chat History 저장 및 입력:** Chat History를 저장하고 적용하는 기능을 구현합니다.

## 0. 환경 설정
- 필요한 라이브러리를 불러옵니다.

In [ ]:
import contextlib
import io
import os
import time

import pandas as pd
from IPython.display import Image, display
from langchain.document_loaders import PyPDFLoader
from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langgraph.graph import END, StateGraph
from typing_extensions import TypedDict
from typing import Any, Dict, List, Optional, Tuple

- Ollama를 통해 llama 3.1 8B 모델을 불러옵니다.

In [ ]:
!ollama pull llama3.1

In [ ]:
llm = ChatOllama(model="llama3.1")
route_llm = ChatOllama(model="llama3.1", format="json")
embeddings = OllamaEmbeddings(model="llama3.1")

## 1. Chat History 저장 및 적용

챗봇은 기본적으로 이전 대화 내용을 기억하지 않습니다. 즉, 유저가 자신의 이름을 말하거나 이전 질문에 이어지는 질문을 해도 챗봇은 이를 기억하지 못하고 대화를 이해할 수 있는 능력이 떨어집니다.

저희가 프로젝트에서 구현하는 챗봇은 이러한 기억 능력이 없어도 필요한 정보를 충분히 Retrieve 할 수 있지만, Chat History를 기억해야 하는 다른 챗봇을 구현할 때는 문제가 될 수 있습니다.

LangGraph는 자체적으로 메모리 저장 기능을 가지고 있습니다. 이번 실습에서는 LangGraph 메모리 기능을 활용하여 챗봇이 대화 내용을 기억할 수 있도록 해보겠습니다.

이번 실습의 목표 챗봇을 구현하기 앞서, 간단한 질의응답 체인으로 Chat History가 저장되지 않는 현상을 재현해봅시다.

In [ ]:
messages_with_variables = [
    ("system", "당신은 친절한 AI Assistant 입니다."),
    ("human", "{question}"),
]
parser = StrOutputParser()
prompt = ChatPromptTemplate.from_messages(messages_with_variables)
chain = prompt | llm | parser

In [ ]:
response = chain.invoke("챗봇을 만드는 방법을 설명해줘.")
print(response)

In [ ]:
response = chain.invoke("방금 말한 것을 영어로 번역해줘")
print(response)

간단한 질문에 잘 대답하는 것을 볼 수 있지만, 답변한 내용에 기반해서 다시 물어보면 뜬금없는 이야기를 하는 것을 볼 수 있습니다.

### 1.1 엑셀 데이터와 PDF 문서를 활용하는 챗봇 구현
LangGraph가 자체적으로 가지고 있는 메모리 기능을 사용하도록 설정해 봅시다. 이를 위해 엑셀 데이터와 PDF 문서를 활용하는 챗봇을 다시 구현해 보겠습니다.

1. Vector DB를 정의하고, 유틸리티성 함수를 정의합니다.

In [ ]:
excel_data_name = "한국지능정보사회진흥원_인공지능 학습용 데이터 구축 현황_20210104.csv"
pdf_data_name = "RE177_2023년 국내외 인공지능 산업 동향 연구_2장.pdf"

# 데이터를 불러오고, 이름과 컬럼명을 저장합니다.
data_dir = "./data"
df_ai_train_data_dist = pd.read_csv(
    os.path.join(data_dir, excel_data_name), index_col=None
)

# 데이터를 저장한 변수명을 LLM에 제공하여 이 변수를 활용하는 코드를 작성하게 할 수 있습니다.
df_name = "df_ai_train_data_dist"
df_columns = ", ".join(df_ai_train_data_dist.columns)


# 시장 조사 문건을 불러옵니다.
doc_path = os.path.join(data_dir, pdf_data_name)
loader = PyPDFLoader(doc_path)
docs = loader.load()

# FAISS 또는 Chroma를 사용하여 문서를 벡터화합니다.
# vectorstore_faiss = FAISS.load_local(
#     "./vectorstore/faiss",
#     embeddings=embeddings,
#     allow_dangerous_deserialization=True,
# )
# db_retriever = new_vectorstore_faiss.as_retriever()

# ChromaDB는 Vector DB를 생성할 때 persist_directory인자를 지정하면 해당 디렉토리에 저장됩니다.
vectorstore = Chroma(
    embedding_function=embeddings, persist_directory="./vectorstore/chroma"
)

db_retriever = vectorstore.as_retriever()


# LLM이 생성한 코드를 파싱하는 함수를 정의합니다.
def python_code_parser(input: str) -> str:
    # LLM은 대부분 ``` 블럭 안에 코드를 출력합니다. 이를 활용합니다.
    # ```python (코드) ```, 혹은 ``` (코드) ``` 형태로 출력됩니다. 두 경우 모두에 대응하도록 코드를 작성합니다.
    processed_input = input.replace("```python", "```").strip()
    parsed_input_list = processed_input.split("```")

    # 만약 ``` 블럭이 없다면, 입력 텍스트 전체가 코드라고 간주합니다.
    # 아닐 경우 이어지는 코드 실행 과정에서 예외 처리를 통해 오류를 확인할 수 있습니다.
    if len(parsed_input_list) == 1:
        return processed_input

    # 코드 부분만 추출합니다.
    # LLM은 여러 코드 블럭에 걸쳐 필요한 코드를 출력할 수 있으므로, 코드가 있는 홀수 번째 텍스트를 모두 저장합니다.
    parsed_code_list = []
    for i in range(1, len(parsed_input_list), 2):
        parsed_code_list.append(parsed_input_list[i])

    # 코드 부분을 하나로 합칩니다.
    return "\n".join(parsed_code_list)


# 생성한 코드를 실행하는 함수를 정의합니다.
def run_code(input_code: str):
    # 코드가 출력한 값을 캡쳐하기 위한 StringIO 객체를 생성합니다.
    output = io.StringIO()
    try:
        # Redirect stdout to the StringIO object
        with contextlib.redirect_stdout(output):
            # Python 3.10 버전이므로, 키워드 인자를 사용할 수 없습니다.
            # 코드가 실행하면서 출력한 모든 결과를 캡쳐합니다.
            exec(input_code, {"df_ai_train_data_dist": df_ai_train_data_dist})
    except Exception as e:
        # 에러가 발생할 경우, 이를 StringIO 객체에 저장합니다.
        print(f"Error: {e}", file=output)
    # StringIO 객체에 저장된 값을 반환합니다.
    return output.getvalue()

2. 그래프 객체를 생성합니다.

In [ ]:
class State(TypedDict):
    # 그래프 상태의 속성을 정의합니다.
    # 질문, LLM이 생성한 텍스트, 데이터, 코드를 저장합니다.
    question: str
    generation: str
    data: str
    code: str
    context: List[str]  # LLM이 생성한 텍스트를 저장합니다. 1.3챕터에서 설명합니다.


# 그래프를 구성하기 위해 StateGraph 객체를 생성합니다.
# 생성자의 인자로 State를 전달하여 Node 간에 정보를 전달할 때 State type을 사용함을 명시합니다.
workflow = StateGraph(State)

3. 각 Node에 대응하는 기능을 구현합니다.

In [ ]:
def query(state: State):
    """
    데이터를 쿼리하는 코드를 생성하고, 실행하고, 그 결과를 포함한 State를 반환합니다.
    위 과정은 앞서 정의한 `find_data` 함수를 활용합니다.

    Args:
        state (dict): 현재 그래프 상태

    Returns:
        state (dict): 쿼리한 데이터를 포함한 새로운 State
    """
    print("---데이터 쿼리---")  # 현재 상태를 확인하기 위한 Print문
    question = state["question"]

    # Retrieval
    # 이전 실습에서 `find_data` 함수를 사용했지만, 여기서는 query 함수에 해당 로직을 포함시켰습니다.
    system_message = "당신은 주어진 데이터를 분석하는 데이터 분석가입니다.\n"
    system_message += f"주어진 DataFrame에서 데이터를 출력하여 주어진 질문에 답할 수 있는 파이썬 코드를 작성하세요. "
    system_message += f"{df_name} DataFrame에 액세스할 수 있습니다.\n"
    system_message += (
        f"`{df_name}` DataFrame에는 다음과 같은 열이 있습니다: {df_columns}\n"
    )
    system_message += (
        "데이터는 이미 로드되어 있으므로 데이터 로드 코드를 생략해야 합니다."
    )

    message_with_data_info = [
        ("system", system_message),
        ("human", "{question}"),
    ]

    prompt_with_data_info = ChatPromptTemplate.from_messages(message_with_data_info)

    # 체인을 구성합니다.
    code_generate_chain = (
        {"question": RunnablePassthrough()}
        | prompt_with_data_info
        | llm
        | StrOutputParser()
        | python_code_parser
    )
    code = code_generate_chain.invoke(question)
    data = run_code(code)
    return {"question": question, "code": code, "data": data, "generation": code}


def answer_with_data(state: State):
    """
    쿼리한 데이터를 바탕으로 답변을 생성합니다.

    Args:
        state (dict): 현재 그래프 상태

    Returns:
        state (dict): LLM의 답변을 포함한 새로운 State
    """
    print("---데이터 기반 답변 생성---")  # 현재 상태를 확인하기 위한 Print문
    question = state["question"]
    data = state["data"]

    # 데이터를 바탕으로 질문에 대답하는 코드를 생성합니다.
    reasoning_system_message = (
        "당신은 데이터를 바탕으로 질문에 답하는 데이터 분석가입니다.\n"
    )
    reasoning_system_message += f"사용자가 입력한 데이터를 바탕으로, 질문에 대답하세요."

    reasoning_user_message = "데이터: {data}\n{question}"

    reasoning_with_data = [
        ("system", reasoning_system_message),
        ("human", reasoning_user_message),
    ]
    reasoning_with_data_chain = (
        ChatPromptTemplate.from_messages(reasoning_with_data) | llm | StrOutputParser()
    )

    # 대답 생성
    generation = reasoning_with_data_chain.invoke({"data": data, "question": question})
    return {
        "question": question,
        "code": state["code"],
        "data": data,
        "generation": generation,
    }


def answer(state: State):
    """
    데이터를 쿼리하지 않고 답변을 바로 생성합니다.

    Args:
        state (dict): 현재 그래프 상태

    Returns:
        state (dict): LLM의 답변을 포함한 새로운 State
    """
    print("---답변 생성---")  # 현재 상태를 확인하기 위한 Print문
    question = state["question"]

    return {"question": question, "generation": llm.invoke(question).content}


def plot_graph(state: State):
    """
    현재 그래프 상태를 시각화합니다.

    Args:
        state (dict): 현재 그래프 상태

    Returns:
        None
    """
    print("---그래프 시각화---")  # 현재 상태를 확인하기 위한 Print문
    question = state["question"]

    # Draw Graph
    system_message = "당신은 주어진 데이터를 분석하는 데이터 분석가입니다.\n"
    system_message += f"주어진 DataFrame에서 데이터를 추출하여 사용자의 질문에 답할 수 있는 그래프를 그리는 코드를 작성하세요. "
    system_message += f"{df_name} DataFrame에 액세스할 수 있습니다.\n"
    system_message += (
        f"`{df_name}` DataFrame에는 다음과 같은 열이 있습니다: {df_columns}\n"
    )
    system_message += (
        "데이터는 이미 로드되어 있으므로 데이터 로드 코드를 생략해야 합니다."
    )

    message_with_data_info = [
        ("system", system_message),
        ("human", "{question}"),
    ]

    prompt_with_data_info = ChatPromptTemplate.from_messages(message_with_data_info)

    # 체인을 구성합니다.
    code_generate_chain = (
        {"question": RunnablePassthrough()}
        | prompt_with_data_info
        | llm
        | StrOutputParser()
        | python_code_parser
    )
    code = code_generate_chain.invoke(question)
    answer = run_code(code)
    return {"question": question, "code": code, "data": answer, "generation": code}


def retrieval(state: State):
    """
    데이터 검색을 수행합니다.

    Args:
        state (dict): 현재 그래프 상태

    Returns:
        state (dict): 검색된 데이터를 포함한 새로운 State
    """

    def get_retrieved_text(docs):
        result = "\n".join([doc.page_content for doc in docs])
        return result

    print("---데이터 검색---")  # 현재 상태를 확인하기 위한 Print문
    question = state["question"]

    # Retrieval Chain
    retrieval_chain = db_retriever | get_retrieved_text

    data = retrieval_chain.invoke(question)

    return {"question": question, "data": data}


def answer_with_retrieved_data(state: State):
    """
    검색된 데이터를 바탕으로 답변을 생성합니다.

    Args:
        state (dict): 현재 그래프 상태

    Returns:
        state (dict): LLM의 답변을 포함한 새로운 State
    """

    print(
        "---검색된 데이터를 바탕으로 답변 생성---"
    )  # 현재 상태를 확인하기 위한 Print문

    question = state["question"]
    data = state["data"]

    messages_with_contexts = [
        ("system", "사용자가 입력하는 정보를 바탕으로 질문에 답하세요."),
        ("human", "정보: {context}.\n{question}."),
    ]
    prompt_with_context = ChatPromptTemplate.from_messages(messages_with_contexts)

    # 체인 구성
    qa_chain = prompt_with_context | llm | StrOutputParser()

    generation = qa_chain.invoke({"context": data, "question": question})
    return {"question": question, "data": data, "generation": generation}

    # 시스템 메시지에 사용 가능한 툴과 각 툴을 사용할 상황을 명시합니다.


# 수월한 선택을 위해 JSON 형식으로 출력하도록 프롬프트에 지정합니다.
route_system_message = """당신은 사용자의 질문에 RAG, 엑셀 데이터 중 어떤 것을 활용할 수 있는지 결정하는 전문가입니다.
인공지능 산업 동향과 관련된 질문이라면 RAG를 활용하세요.
인공지능 데이터 프로필과 관련된 질문이라면 excel_data를 활용하세요.
그래프를 그리라는 질문이면 excel_plot을 활용하세요.
둘 다 아니라면, plain_answer로 충분합니다.
주어진 질문에 맞춰 `rag`, `excel_data`, `excel_plot`, `plain_answer`중 하나를 선택하세요.
답변은 `route` key 하나만 있는 JSON으로 답변하고, 다른 텍스트나 설명을 생성하지 마세요."""
route_user_message = "{question}"
route_prompt = ChatPromptTemplate.from_messages(
    [("system", route_system_message), ("human", route_user_message)]
)

# 로직 선택용 ChatOllama 객체를 생성합니다. format="json" 인자를 적용하여 출력 양식을 json으로 강제합니다.
# 같은 질문에 항상 같은 대답을 유도하기 위해 temperature를 0으로 설정합니다.
route_llm = ChatOllama(model="llama3.1", format="json", temperature=0)
router_chain = route_prompt | route_llm | JsonOutputParser()


def init_answer(state: State) -> str:
    """
    초기 질문의 경로를 결정합니다.
    """
    question = state["question"]
    route = router_chain.invoke({"question": question})["route"]
    return {"question": question, "generation": route}


def route_question(state: State) -> str:
    route = state["generation"]
    return route.lower().strip()

4. 그래프를 구성합니다.

In [ ]:
## 그래프 구성

# 앞서 정의한 Node를 모두 추가합니다.
workflow.add_node("init_answer", init_answer)

workflow.add_node("excel_data", query)
workflow.add_node("rag", retrieval)

workflow.add_node("excel_plot", plot_graph)
workflow.add_node("answer_with_data", answer_with_data)
workflow.add_node("plain_answer", answer)
workflow.add_node("answer_with_retrieval", answer_with_retrieved_data)

# 시작지점을 정의합니다.
workflow.set_entry_point("init_answer")

# 간선을 정의합니다.
# END는 종결 지점을 의미합니다.
workflow.add_edge(
    "plain_answer", END
)  # workflow.set_finish_point("answer")와 동일합니다.
workflow.add_edge("answer_with_data", END)
workflow.add_edge("answer_with_retrieval", END)
workflow.add_edge("excel_plot", END)  # 그래프를 그리고 종결합니다.
workflow.add_edge("excel_data", "answer_with_data")
workflow.add_edge("rag", "answer_with_retrieval")


# 조건부 간선을 정의합니다.
# init_answer 노드의 답변을 바탕으로 decide_query 함수에서 query 또는 answer로 분기합니다.
workflow.add_conditional_edges(
    "init_answer",
    route_question,
    # 어떤 노드로 이동할지 mapping합니다. 없어도 무방하지만, Graph의 가독성을 높일 수 있습니다.
    {
        "excel_data": "excel_data",
        "rag": "rag",
        "excel_plot": "excel_plot",
        "plain_answer": "plain_answer",
    },
)

### 1.2 MemorySaver

그래프 구조를 컴파일하기 앞서 메모리에 정보를 저장하는 `MemorySaver` 객체를 생성합니다. 이 객체를 활용하여 메모리 상에 현재 상태를 저장하고, 다음 입력 시 현재 상태를 불러와서 활용해 보겠습니다. 

Note. `MemorySaver`는 프로그램이 종료되거나 기타 사유로 메모리가 손상되면 정보가 모두 사라지는 단점이 있습니다. 이로 인해 LangGraph에서는 프로덕션 레벨에서는 메모리에 정보를 저장하는 `MemorySaver` 대신 Postgre DB 시스템을 활용하는 `PostgresSaver`를 사용하는 것을 권장합니다. [출처 링크](https://langchain-ai.github.io/langgraph/reference/checkpoints/#langgraph.checkpoint.memory.MemorySaver)

그래프를 컴파일하기 앞서 `MemorySaver` 객체를 생성합니다. 

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# Add memory
memory = MemorySaver()
graph = workflow.compile(checkpointer=memory)

`MemorySaver`는 여러 사용자 별로 현재 상태를 모두 저장합니다. 이를 식별하기 위해 임의의 ID를 아래 코드의 `config` 변수와 같이 설정할 수 있습니다.

In [ ]:
config = {"configurable": {"thread_id": "user_A"}}

config를 설정한 후, 간단한 대화를 진행해 봅시다.

In [ ]:
while True:
    question = input("질문을 입력해주세요 (종료를 원하시면 '종료'를 입력해주세요.): ")
    if question == "종료":
        break
    else:
        # graph.invoke 함수를 사용하여 그래프를 실행하고, 최종 결과를 반환합니다.
        # 답변 생성에는 약 1분 정도 소요됩니다.
        print(
            "Assistant: ",
            graph.invoke({"question": question}, config=config)["generation"],
        )

대화를 마친 후 저장된 State를 모두 확인해봅시다.

In [ ]:
history = graph.get_state_history(config=config)
for i, state in enumerate(history):
    print(f"State {i}: ", state)

State가 저장되긴 하지만 여전히 대화가 이뤄지지 않는 것을 확인할 수 있습니다. 이는 저장된 State 값을 대화에 활용하지 않아서 생기는 문제입니다.

현재 그래프 구조를 살짝 변경하여, 이전 대화 기록도 모두 고려하는 형태로 변환해봅시다.

앞서 저희가 정의한 `State` Class 구현체를 확인해보면, 새로운 Field가 추가된 것을 알 수 있습니다.
```python
class State(TypedDict):
    # 그래프 상태의 속성을 정의합니다.
    # 질문, LLM이 생성한 텍스트, 데이터, 코드를 저장합니다.
    question: str
    generation: str
    data: str
    code: str
    context: List[str] # LLM이 생성한 텍스트를 저장합니다. 1.3챕터에서 설명합니다.
```
이 context 값에 현재까지의 대화를 모두 저장하고, 기존 함수에 context 값을 활용하는 기능을 추가하면 문제 없이 State를 활용할 수 있을 것입니다.

context 값을 확인하는 함수를 구현해 봅시다.

In [ ]:
def query_with_context(state: State):
    """
    데이터를 쿼리하는 코드를 생성하고, 실행하고, 그 결과를 포함한 State를 반환합니다.
    위 과정은 앞서 정의한 `find_data` 함수를 활용합니다.

    Args:
        state (dict): 현재 그래프 상태

    Returns:
        state (dict): 쿼리한 데이터를 포함한 새로운 State
    """
    print("---데이터 쿼리---")  # 현재 상태를 확인하기 위한 Print문
    question = state["question"]
    context = state["context"]

    context.append(("human", f"{question}"))

    # Retrieval
    # 이전 실습에서 `find_data` 함수를 사용했지만, 여기서는 query 함수에 해당 로직을 포함시켰습니다.
    system_message = "당신은 주어진 데이터를 분석하는 데이터 분석가입니다.\n"
    system_message += f"주어진 DataFrame에서 데이터를 출력하여 주어진 질문에 답할 수 있는 파이썬 코드를 작성하세요. "
    system_message += f"{df_name} DataFrame에 액세스할 수 있습니다.\n"
    system_message += (
        f"`{df_name}` DataFrame에는 다음과 같은 열이 있습니다: {df_columns}\n"
    )
    system_message += (
        "데이터는 이미 로드되어 있으므로 데이터 로드 코드를 생략해야 합니다."
    )

    message_with_data_info = [
        ("system", system_message),
    ]

    message_with_data_info.extend(context)
    message_with_data_info.append(("human", "{question}"))
    prompt_with_data_info = ChatPromptTemplate.from_messages(message_with_data_info)

    # 체인을 구성합니다.
    code_generate_chain = (
        prompt_with_data_info | llm | StrOutputParser() | python_code_parser
    )
    code = code_generate_chain.invoke({"question": question})
    data = run_code(code)
    context.append(("assistant", "\n".join([code, f"Result: {data}"])))

    return {
        "question": question,
        "code": code,
        "data": data,
        "generation": code,
        "context": context,
    }


def answer_with_data_and_context(state: State):
    """
    쿼리한 데이터를 바탕으로 답변을 생성합니다.

    Args:
        state (dict): 현재 그래프 상태

    Returns:
        state (dict): LLM의 답변을 포함한 새로운 State
    """
    print("---데이터 기반 답변 생성---")  # 현재 상태를 확인하기 위한 Print문
    question = state["question"]
    data = state["data"]
    context = state["context"]
    context.append(("human", f"데이터: {data}\n{question}"))

    # 데이터를 바탕으로 질문에 대답하는 코드를 생성합니다.
    reasoning_system_message = (
        "당신은 데이터를 바탕으로 질문에 답하는 데이터 분석가입니다.\n"
    )
    reasoning_system_message += f"사용자가 입력한 데이터를 바탕으로, 질문에 대답하세요."

    reasoning_with_data = [
        ("system", reasoning_system_message),
    ]
    reasoning_with_data.extend(context)
    prompt_with_data_info = ChatPromptTemplate.from_messages(reasoning_with_data)

    reasoning_with_data_chain = llm | StrOutputParser()

    # 대답 생성
    generation = reasoning_with_data_chain.invoke(prompt_with_data_info)
    context.append(("assistant", generation))
    return {
        "question": question,
        "code": state["code"],
        "data": data,
        "generation": generation,
        "context": context,
    }


def answer_with_context(state: State):
    """
    데이터를 쿼리하지 않고 답변을 바로 생성합니다.

    Args:
        state (dict): 현재 그래프 상태

    Returns:
        state (dict): LLM의 답변을 포함한 새로운 State
    """
    print("---답변 생성---")  # 현재 상태를 확인하기 위한 Print문
    context = state["context"]
    question = state["question"]
    messages = context + [("human", "{question}")]
    context.append(("human", f"{question}"))
    prompt = ChatPromptTemplate.from_messages(messages)

    ask_chain = prompt | llm | StrOutputParser()

    answer = ask_chain.invoke({"question": question})
    print(type(answer))
    context.append(("assistant", answer))

    return {
        "question": question,
        "generation": answer,
        "context": context,
    }


def plot_graph(state: State):
    """
    현재 그래프 상태를 시각화합니다.

    Args:
        state (dict): 현재 그래프 상태

    Returns:
        None
    """
    print("---그래프 시각화---")  # 현재 상태를 확인하기 위한 Print문
    question = state["question"]
    context = state["context"]

    context.append(("human", f"{question}"))

    # Draw Graph
    system_message = "당신은 주어진 데이터를 분석하는 데이터 분석가입니다.\n"
    system_message += f"주어진 DataFrame에서 데이터를 추출하여 사용자의 질문에 답할 수 있는 그래프를 그리는 코드를 작성하세요. "
    system_message += f"{df_name} DataFrame에 액세스할 수 있습니다.\n"
    system_message += (
        f"`{df_name}` DataFrame에는 다음과 같은 열이 있습니다: {df_columns}\n"
    )
    system_message += (
        "데이터는 이미 로드되어 있으므로 데이터 로드 코드를 생략해야 합니다."
    )

    message_with_data_info = [("system", system_message)]
    message_with_data_info.extend(context)

    prompt_with_data_info = ChatPromptTemplate.from_messages(message_with_data_info)

    # 체인을 구성합니다.
    code_generate_chain = (
        prompt_with_data_info | llm | StrOutputParser() | python_code_parser
    )
    code = code_generate_chain.invoke(question)
    answer = run_code(code)
    context.append(("assistant", answer))
    return {
        "question": question,
        "code": code,
        "data": answer,
        "generation": code,
        "context": context,
    }


def retrieval_with_context(state: State):
    """
    데이터 검색을 수행합니다.

    Args:
        state (dict): 현재 그래프 상태

    Returns:
        state (dict): 검색된 데이터를 포함한 새로운 State
    """

    def get_retrieved_text(docs):
        result = "\n".join([doc.page_content for doc in docs])
        return result

    print("---데이터 검색---")  # 현재 상태를 확인하기 위한 Print문
    question = state["question"]
    context = state["context"]
    messages = context + [("human", "{question}")]

    # Retrieval Chain
    retrieval_chain = db_retriever | get_retrieved_text

    data = retrieval_chain.invoke(question)

    context.append(("human", f"{question}"))
    context.append(("assistant", f"검색 결과: {data}"))

    return {"question": question, "data": data, "context": context}


def answer_with_retrieved_data_and_context(state: State):
    """
    검색된 데이터를 바탕으로 답변을 생성합니다.

    Args:
        state (dict): 현재 그래프 상태

    Returns:
        state (dict): LLM의 답변을 포함한 새로운 State
    """

    print(
        "---검색된 데이터를 바탕으로 답변 생성---"
    )  # 현재 상태를 확인하기 위한 Print문

    question = state["question"]
    data = state["data"]
    context = state["context"]

    messages_with_contexts = [
        ("system", "사용자가 입력하는 정보를 바탕으로 질문에 답하세요."),
    ]
    messages_with_contexts.extend(context)
    messages_with_contexts.append(("human", "정보: {data}\n{question}"))
    prompt_with_context = ChatPromptTemplate.from_messages(messages_with_contexts)

    # 체인 구성
    qa_chain = prompt_with_context | llm | StrOutputParser()

    generation = qa_chain.invoke({"data": data, "question": question})

    context.append(("human", f"정보: {data}\n{question}"))
    context.append(("assistant", generation))

    return {
        "question": question,
        "data": data,
        "generation": generation,
        "context": context,
    }

def init_answer_with_context(state: State) -> str:
    "초기 질문의 경로를 결정합니다."
    question = state["question"]
    context = state["context"]
    route_system_message = """당신은 사용자의 질문에 RAG, 엑셀 데이터 중 어떤 것을 활용할 수 있는지 결정하는 전문가입니다.
인공지능 산업 동향과 관련된 질문이라면 RAG를 활용하세요.
인공지능 데이터 프로필과 관련된 질문이라면 excel_data를 활용하세요.
그래프를 그리라는 질문이면 excel_plot을 활용하세요.
둘 다 아니라면, plain_answer로 충분합니다.
주어진 질문에 맞춰 `rag`, `excel_data`, `excel_plot`, `plain_answer`중 하나를 선택하세요.
답변은 `route` key 하나만 있는 JSON으로 답변하고, 다른 텍스트나 설명을 생성하지 마세요."""

    message_list = [("system", route_system_message)]
    message_list.extend(context)
    print(context)
    message_list.append(("human", "{question}"))

    route_prompt = ChatPromptTemplate.from_messages(message_list)

    # 로직 선택용 ChatOllama 객체를 생성합니다. format="json" 인자를 적용하여 출력 양식을 json으로 강제합니다.
    # 같은 질문에 항상 같은 대답을 유도하기 위해 temperature를 0으로 설정합니다.
    router_chain = route_prompt | route_llm | JsonOutputParser()
    prompt = ChatPromptTemplate.from_messages({"question": question})

    route = router_chain.invoke(prompt)["route"]

    return {"question": question, "generation": route, "context": context}

새로 정의한 함수를 활용해서 다른 Graph를 정의합니다.

In [ ]:
## 그래프 구성
workflow_with_context = StateGraph(State)

# 앞서 정의한 Node를 모두 추가합니다.
workflow_with_context.add_node("init_answer", init_answer_with_context)

workflow_with_context.add_node("excel_data", query_with_context)
workflow_with_context.add_node("rag", retrieval_with_context)

workflow_with_context.add_node("excel_plot", plot_graph)
workflow_with_context.add_node("answer_with_data", answer_with_data_and_context)
workflow_with_context.add_node("plain_answer", answer_with_context)
workflow_with_context.add_node(
    "answer_with_retrieval", answer_with_retrieved_data_and_context
)

# 시작지점을 정의합니다.
workflow_with_context.set_entry_point("init_answer")

# 간선을 정의합니다.
# END는 종결 지점을 의미합니다.
workflow_with_context.add_edge(
    "plain_answer", END
)  # workflow.set_finish_point("answer")와 동일합니다.
workflow_with_context.add_edge("answer_with_data", END)
workflow_with_context.add_edge("answer_with_retrieval", END)
workflow_with_context.add_edge("excel_plot", END)  # 그래프를 그리고 종결합니다.
workflow_with_context.add_edge("excel_data", "answer_with_data")
workflow_with_context.add_edge("rag", "answer_with_retrieval")


# 조건부 간선을 정의합니다.
# init_answer 노드의 답변을 바탕으로 decide_query 함수에서 query 또는 answer로 분기합니다.
workflow_with_context.add_conditional_edges(
    "init_answer",
    route_question,
    # 어떤 노드로 이동할지 mapping합니다. 없어도 무방하지만, Graph의 가독성을 높일 수 있습니다.
    {
        "excel_data": "excel_data",
        "rag": "rag",
        "excel_plot": "excel_plot",
        "plain_answer": "plain_answer",
    },
)

In [ ]:
graph_with_context = workflow_with_context.compile(checkpointer=memory)

In [ ]:
last_context = []

while True:
    question = input("질문을 입력해주세요 (종료를 원하시면 '종료'를 입력해주세요.): ")
    if question == "종료":
        break
    else:
        # graph.invoke 함수를 사용하여 그래프를 실행하고, 최종 결과를 반환합니다.
        # 답변 생성에는 약 1분 정도 소요됩니다.
        print(
            "Assistant: ",
            graph_with_context.invoke(
                {"question": question, "context": last_context}, config=config
            )["generation"],
        )
        # Contetxt가 Tuple list가 아니라 이상하게 저장되는 현상 확인 필요.
        print("State: ", graph_with_context.get_state(config=config))
        last_context = graph_with_context.get_state(config=config).values["context"]
        # Change list to tuple
        last_context = [tuple(i) for i in last_context]

LangGraph의 메모리 기능을 활용해서 현재까지의 대화 내용 및 기타 정보를 저장하고, 이를 다시 입력함으로써 이전 대화까지 잘 고려하는 것을 확인할 수 있습니다.

엑셀 데이터 출처: 한국지능정보사회진흥원_인공지능 학습용 데이터 구축 현황
  - https://www.data.go.kr/data/15039915/fileData.do 

PDF 문서 출처: 소프트웨어정책연구소 (SRPI) 2023년 국내외 인공지능 산업 동향 연구 보고서 (일부 발췌)
  - 본 과정에서는 전체 보고서 중 제2장 인공지능 산업 현황 및 전망 구간만 발췌하여 사용합니다.
  - https://spri.kr/posts/view/23728?code=research&study_type=&board_type=&flg=#none